# Estimativa de calorias de itens do iFood

Lê o JSON de um item salvo pelo agente (opencode + MCP do iFood) e descobre as calorias em duas etapas:

1. **Busca online:** o **GPT-5.4-Mini** procura na web a tabela nutricional publicada pelo restaurante (ou rede) para esse produto. Se encontrar uma fonte confiável, usa esses valores e pula a análise da imagem.
2. **Estimativa:** se não achar, junta nome, restaurante, descrição, imagem e opções escolhidas e pede ao modelo uma estimativa com valor médio, faixa mínima e máxima e nível de confiança.

In [31]:
%pip install -q openai requests python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Configuração

In [32]:
import base64
import json
import os
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse

import openai
import requests
from IPython.display import Image, Markdown, display
from dotenv import load_dotenv
from openai import OpenAI

ARQUIVO_PEDIDO = Path("pedido_teste.json")
MODELO = "gpt-5.4-mini"

# Se True, procura a tabela nutricional do restaurante na web antes de estimar.
BUSCAR_ONLINE = True

# Lê OPENAI_API_KEY do arquivo .env na pasta do projeto; se não houver, pede a chave.
load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

client = OpenAI()

## 1. Carregar o item

In [33]:
with open(ARQUIVO_PEDIDO, encoding="utf-8") as f:
    item_bruto = json.load(f)

item_bruto

{'itemName': 'Pizza Grande',
 'merchantName': "Domino's Pizza - Chácara Primavera",
 'itemId': 'ce44c408-ac0c-46a2-aea8-b25641f96b41',
 'itemImageUrl': 'https://static.ifood-static.com.br/image/upload/t_medium/pratos/94dbe494-0fe5-42c0-85fd-261c33785c3c/202605141258_52JL_i.jpg',
 'itemDescription': 'Pizza Grande (35 cm) com 8 fatias. Massa saborosa, recheio caprichado e ingredientes de qualidade para tornar cada fatia irresistível. Perfeita para compartilhar!',
 'price': {'unitPrice': 104.9,
  'promotionalUnitPrice': 104.9,
  'currency': 'BRL'},
 'merchantMinimumOrderAmount': 49,
 'productInformation': {'optionGroups': [{'optionGroupName': 'Escolha sua massa',
    'optionGroupId': '134fbb6d-6aff-4941-b4fd-d637466009cb',
    'minSelections': 1,
    'maxSelections': 1,
    'selected': {'optionName': 'Massa Tradicional',
     'optionId': '4ac5283a-ea99-43ed-adc4-0ea53c57dd98',
     'additionalPrice': 0,
     'currency': 'BRL'}},
   {'optionGroupName': 'Escolha o primeiro sabor',
    'opti

## 2. Extrair as características do produto

Os grupos de opções podem ter uma única escolha (`selected` é um objeto) ou várias (`selected` é uma lista). A função trata os dois casos e ignora grupos sem escolha.

In [34]:
def extrair_opcoes(produto: dict) -> list[dict]:
    grupos = (produto.get("productInformation") or {}).get("optionGroups") or []
    opcoes = []
    for grupo in grupos:
        selecionado = grupo.get("selected")
        if not selecionado:
            continue
        if isinstance(selecionado, dict):
            selecionado = [selecionado]
        for opcao in selecionado:
            opcoes.append({
                "grupo": grupo.get("optionGroupName"),
                "opcao": opcao.get("optionName"),
                "quantidade": opcao.get("quantity", 1),
            })
    return opcoes


def normalizar_item(produto: dict) -> dict:
    restaurante = (
        produto.get("merchantName")
        or (produto.get("merchant") or {}).get("name")
        or "não informado"
    )
    return {
        "nome": produto.get("itemName", ""),
        "restaurante": restaurante,
        "descricao": produto.get("itemDescription", ""),
        "imagem_url": produto.get("itemImageUrl"),
        "opcoes": extrair_opcoes(produto),
    }


item = normalizar_item(item_bruto)
item

{'nome': 'Pizza Grande',
 'restaurante': "Domino's Pizza - Chácara Primavera",
 'descricao': 'Pizza Grande (35 cm) com 8 fatias. Massa saborosa, recheio caprichado e ingredientes de qualidade para tornar cada fatia irresistível. Perfeita para compartilhar!',
 'imagem_url': 'https://static.ifood-static.com.br/image/upload/t_medium/pratos/94dbe494-0fe5-42c0-85fd-261c33785c3c/202605141258_52JL_i.jpg',
 'opcoes': [{'grupo': 'Escolha sua massa',
   'opcao': 'Massa Tradicional',
   'quantidade': 1},
  {'grupo': 'Escolha o primeiro sabor',
   'opcao': 'Cheddar & Bacon',
   'quantidade': 1},
  {'grupo': 'Escolha o segundo sabor', 'opcao': 'Pepperoni', 'quantidade': 1}]}

In [35]:
if item["imagem_url"]:
    display(Image(url=item["imagem_url"], width=350))

## 2b. Montar o prompt e o formato da resposta

Precisa rodar antes da busca online, porque ela reaproveita o `montar_prompt`. A resposta usa *Structured Outputs* (JSON Schema estrito), então o modelo sempre devolve um JSON válido nesse formato, sem precisar limpar markdown.

In [36]:
INSTRUCOES = """Você é um nutricionista que estima calorias de pratos de delivery.
Use o nome, o restaurante, a descrição, as opções escolhidas e a imagem (quando houver). O peso da avaliação da imagem deve ser menor que o da descrição, mas pode ajudar a identificar ingredientes e tamanho da porção.
Considere o tamanho da porção (informado na descrição ou aparente na imagem), os ingredientes
de cada opção e valores típicos de redes conhecidas quando o restaurante for uma delas.
A estimativa é para o item inteiro, como vendido (todas as porções/fatias).
A faixa mínima e máxima deve refletir a incerteza real; quanto menos informação, mais larga a faixa e menor a confiança."""


def montar_prompt(item: dict) -> str:
    if item["opcoes"]:
        opcoes = "\n".join(
            f"- {o['grupo']}: {o['opcao']}" + (f" (x{o['quantidade']})" if o["quantidade"] != 1 else "")
            for o in item["opcoes"]
        )
    else:
        opcoes = "- nenhuma"
    return f"""Estime as calorias deste item.

Nome: {item['nome']}
Restaurante: {item['restaurante']}
Descrição: {item['descricao']}
Opções escolhidas:
{opcoes}"""


SCHEMA_ESTIMATIVA = {
    "type": "object",
    "properties": {
        "calorias_estimadas": {"type": "integer", "description": "Valor médio estimado em kcal para o item inteiro"},
        "faixa_min": {"type": "integer", "description": "Estimativa mínima em kcal"},
        "faixa_max": {"type": "integer", "description": "Estimativa máxima em kcal"},
        "confianca": {"type": "string", "enum": ["baixa", "media", "alta"]},
        "justificativa": {"type": "string", "description": "1 a 3 frases explicando a estimativa"},
    },
    "required": ["calorias_estimadas", "faixa_min", "faixa_max", "confianca", "justificativa"],
    "additionalProperties": False,
}

print(montar_prompt(item))

Estime as calorias deste item.

Nome: Pizza Grande
Restaurante: Domino's Pizza - Chácara Primavera
Descrição: Pizza Grande (35 cm) com 8 fatias. Massa saborosa, recheio caprichado e ingredientes de qualidade para tornar cada fatia irresistível. Perfeita para compartilhar!
Opções escolhidas:
- Escolha sua massa: Massa Tradicional
- Escolha o primeiro sabor: Cheddar & Bacon
- Escolha o segundo sabor: Pepperoni


## 3. Buscar informação nutricional online

O modelo usa a ferramenta de busca na web da OpenAI para procurar a tabela nutricional do produto no site do restaurante ou da rede. O resultado só é aceito se:

- o modelo disser que encontrou dados **deste produto, deste restaurante**, e não de um produto parecido;
- a URL informada como fonte for de um site que a busca realmente consultou (para evitar fonte inventada).

Caso contrário, `info_online` fica `None` e o notebook segue para a estimativa com imagem.

In [37]:
INSTRUCOES_BUSCA = """Você pesquisa informação nutricional oficial de produtos vendidos por restaurantes no iFood.
Procure na web a tabela nutricional publicada pelo próprio restaurante ou rede (site oficial, cardápio,
PDF de tabela nutricional) ou por uma fonte confiável que reproduza esses dados para ESTE produto.

Regras:
- Só responda encontrado=true se a fonte trouxer as calorias (kcal) do mesmo produto, do mesmo restaurante
  ou rede, e compatível com as opções escolhidas (tamanho, sabor, massa etc.).
- Não use valores de produtos parecidos de outros lugares, tabelas genéricas (ex.: TACO) ou receitas caseiras
  de apps de dieta. Nesses casos responda encontrado=false e deixe os números como null.
- Se a fonte informar calorias por porção (por fatia, por 100 g etc.), converta para o item inteiro como
  vendido e explique a conta na justificativa.
- Se o item combinar opções com valores diferentes (ex.: pizza meio a meio), combine proporcionalmente.
- fonte_url deve ser a URL exata de onde você tirou os números."""

SCHEMA_BUSCA = {
    "type": "object",
    "properties": {
        "encontrado": {"type": "boolean"},
        "calorias_estimadas": {"type": ["integer", "null"], "description": "kcal do item inteiro segundo a fonte"},
        "faixa_min": {"type": ["integer", "null"]},
        "faixa_max": {"type": ["integer", "null"]},
        "confianca": {"type": "string", "enum": ["baixa", "media", "alta"]},
        "fonte_nome": {"type": "string"},
        "fonte_url": {"type": "string"},
        "justificativa": {"type": "string", "description": "1 a 3 frases: o que a fonte informa e como chegou ao total"},
    },
    "required": ["encontrado", "calorias_estimadas", "faixa_min", "faixa_max",
                 "confianca", "fonte_nome", "fonte_url", "justificativa"],
    "additionalProperties": False,
}


def _dominio(url: str) -> str:
    return urlparse(url).netloc.lower().removeprefix("www.")


def _urls_consultadas(resposta) -> set[str]:
    """URLs que a busca na web realmente abriu ou citou nesta resposta."""
    urls = set()
    for saida in resposta.output:
        if saida.type == "web_search_call":
            for fonte in getattr(getattr(saida, "action", None), "sources", None) or []:
                url = fonte.get("url") if isinstance(fonte, dict) else getattr(fonte, "url", None)
                if url:
                    urls.add(url)
        elif saida.type == "message":
            for parte in saida.content:
                for anotacao in getattr(parte, "annotations", None) or []:
                    if getattr(anotacao, "type", None) == "url_citation":
                        urls.add(anotacao.url)
    return urls


def buscar_info_nutricional(item: dict) -> dict | None:
    try:
        resposta = client.responses.create(
            model=MODELO,
            instructions=INSTRUCOES_BUSCA,
            input=montar_prompt(item).replace(
                "Estime as calorias deste item.",
                "Procure a informação nutricional oficial deste item.",
            ),
            tools=[{"type": "web_search"}],
            include=["web_search_call.action.sources"],
            text={"format": {
                "type": "json_schema",
                "name": "busca_nutricional",
                "schema": SCHEMA_BUSCA,
                "strict": True,
            }},
        )
    except openai.APIError as e:
        print(f"Busca online falhou ({e}); seguindo para a estimativa.")
        return None

    resultado = json.loads(resposta.output_text)
    if not resultado["encontrado"] or resultado["calorias_estimadas"] is None:
        print("Não encontrei informação nutricional oficial deste item.")
        print(resultado["justificativa"])
        return None

    dominios_consultados = {_dominio(u) for u in _urls_consultadas(resposta)}
    if _dominio(resultado["fonte_url"]) not in dominios_consultados:
        print(f"Fonte {resultado['fonte_url']} não aparece entre os sites consultados; descartando.")
        return None

    calorias = resultado["calorias_estimadas"]
    return {
        "calorias_estimadas": calorias,
        "faixa_min": min(resultado["faixa_min"] or calorias, calorias),
        "faixa_max": max(resultado["faixa_max"] or calorias, calorias),
        "confianca": resultado["confianca"],
        "justificativa": resultado["justificativa"],
        "metodo": "busca_online",
        "fonte_nome": resultado["fonte_nome"],
        "fonte_url": resultado["fonte_url"],
    }


info_online = buscar_info_nutricional(item) if BUSCAR_ONLINE else None
info_online

Não encontrei informação nutricional oficial deste item.
Encontrei a tabela nutricional oficial da Domino's Brasil, mas ela traz os valores dos ingredientes/toppings separadamente (ex.: Bacon 340 kcal, Cheddar Cheese 230 kcal, Pepperoni 240 kcal) e não do item completo 'Pizza Grande' com massa tradicional e dois sabores combinados. Como o pedido é uma pizza de 35 cm com combinação de sabores, não há na fonte um valor oficial direto do produto exatamente como montado; portanto não dá para afirmar as calorias do item inteiro com segurança.


## 4. Baixar a imagem

Só roda se a busca online não encontrou nada. A imagem é baixada e enviada em base64. Se o download falhar, a estimativa segue só com o texto (e a confiança tende a cair).

In [38]:
def baixar_imagem_data_url(url: str | None) -> str | None:
    if not url:
        return None
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"Não foi possível baixar a imagem: {e}")
        return None
    media_type = resp.headers.get("Content-Type", "image/jpeg").split(";")[0]
    b64 = base64.b64encode(resp.content).decode()
    return f"data:{media_type};base64,{b64}"


if info_online:
    imagem_data_url = None
    print("Informação encontrada online; análise da imagem dispensada.")
else:
    imagem_data_url = baixar_imagem_data_url(item["imagem_url"])
    print("Imagem carregada" if imagem_data_url else "Seguindo sem imagem")

Imagem carregada


## 5. Estimar com o GPT-5.4-Mini

Só chama o modelo se a busca online não encontrou nada.

In [39]:
def estimar_calorias(item: dict, imagem_data_url: str | None = None) -> dict:
    conteudo = [{"type": "input_text", "text": montar_prompt(item)}]
    if imagem_data_url:
        conteudo.append({"type": "input_image", "image_url": imagem_data_url})

    resposta = client.responses.create(
        model=MODELO,
        instructions=INSTRUCOES,
        input=[{"role": "user", "content": conteudo}],
        text={"format": {
            "type": "json_schema",
            "name": "estimativa_calorias",
            "schema": SCHEMA_ESTIMATIVA,
            "strict": True,
        }},
    )
    estimativa = json.loads(resposta.output_text)

    # garante que a faixa contém o valor médio
    estimativa["faixa_min"] = min(estimativa["faixa_min"], estimativa["calorias_estimadas"])
    estimativa["faixa_max"] = max(estimativa["faixa_max"], estimativa["calorias_estimadas"])
    estimativa.update(metodo="estimativa_modelo", fonte_nome=None, fonte_url=None)
    return estimativa


estimativa = info_online or estimar_calorias(item, imagem_data_url)
estimativa

{'calorias_estimadas': 2600,
 'faixa_min': 2300,
 'faixa_max': 2900,
 'confianca': 'alta',
 'justificativa': "Uma pizza grande de 35 cm da Domino's com massa tradicional e metade/combinação de sabores mais calóricos (Cheddar & Bacon e Pepperoni) costuma ficar na faixa de 2300 a 2900 kcal no total. A imagem mostra uma pizza bem recheada, o que reforça uma estimativa em torno de 2600 kcal para a pizza inteira (8 fatias).",
 'metodo': 'estimativa_modelo',
 'fonte_nome': None,
 'fonte_url': None}

## 6. Resultado

In [40]:
if estimativa["metodo"] == "busca_online":
    origem = f"Tabela nutricional encontrada online: [{estimativa['fonte_nome']}]({estimativa['fonte_url']})"
else:
    origem = "Estimativa do modelo (sem tabela nutricional oficial encontrada)"

display(Markdown(f"""
**{item['nome']}** ({item['restaurante']})

{origem}

| | kcal |
|---|---|
| Estimativa média | **{estimativa['calorias_estimadas']}** |
| Mínimo | {estimativa['faixa_min']} |
| Máximo | {estimativa['faixa_max']} |

**Confiança:** {estimativa['confianca']}

{estimativa['justificativa']}
"""))


**Pizza Grande** (Domino's Pizza - Chácara Primavera)

Estimativa do modelo (sem tabela nutricional oficial encontrada)

| | kcal |
|---|---|
| Estimativa média | **2600** |
| Mínimo | 2300 |
| Máximo | 2900 |

**Confiança:** alta

Uma pizza grande de 35 cm da Domino's com massa tradicional e metade/combinação de sabores mais calóricos (Cheddar & Bacon e Pepperoni) costuma ficar na faixa de 2300 a 2900 kcal no total. A imagem mostra uma pizza bem recheada, o que reforça uma estimativa em torno de 2600 kcal para a pizza inteira (8 fatias).


Salva o JSON original com a estimativa anexada, para o agente poder usar depois.

In [41]:
saida = {**item_bruto, "estimativaCalorias": {**estimativa, "modelo": MODELO}}
arquivo_saida = ARQUIVO_PEDIDO.with_name(ARQUIVO_PEDIDO.stem + "_com_calorias.json")
arquivo_saida.write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Salvo em {arquivo_saida}")

Salvo em pedido_teste_com_calorias.json
